# AV1 - Analise e Visualizacao de Dados (2026.2)
## Grupo 6 - EcoVolt | **Notebook 2 de 3 - V2: autonomia x horizonte requerido**

**Objetivo.** Comparar duas alternativas de banco (A e B) contra o tempo de interrupcao requerido em cada cenario, respondendo a pergunta de decisao do grupo.

**Pergunta (V2).** *Qual cenario atende ao horizonte requerido?*

**Estrutura do notebook.** 1) Coleta -> 2) Tratamento -> 3) Limpeza (5 verificacoes) -> 4) Indicador -> 5) Grafico -> 6) Por que este grafico -> 7) Leitura e limitacoes.


## 1. Coleta de dados

Duas fontes, com papeis diferentes:

| Fonte | Papel | Classificacao |
|---|---|---|
| `dados/raw/ORIGINAL_5.596Ah_charge.xlsx` | curva de carga (evidencia de capacidade) | **observado** |
| `analise/cenarios.json` (bancos + fatores) | parametros dos bancos e cenarios | simulado / estimado |


In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

def encontrar_raiz(inicio=None):
    """Sobe a arvore de diretorios ate achar a pasta do projeto (dados/ + analise/)."""
    p = Path(inicio or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "dados").exists() and (cand / "analise").exists():
            return cand
    raise FileNotFoundError("pasta do projeto nao encontrada")


BASE = encontrar_raiz()
RAW = BASE / "dados" / "raw"
PROC = BASE / "dados" / "processados"
VISTAS = BASE / "dashboard" / "vistas"
PROC.mkdir(parents=True, exist_ok=True)
VISTAS.mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", BASE)


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


curva_raw = RAW / "ORIGINAL_5.596Ah_charge.xlsx"
cfg = json.loads((BASE / "analise" / "cenarios.json").read_text(encoding="utf-8"))

print("Curva de carga:", curva_raw.name, "| sha256:", sha256(curva_raw)[:16], "...")
print("Bancos :", [(b["id"], b["Enominal_kWh"], b["Pmax_kW"]) for b in cfg["bancos"]])
print("Fatores:", {k: v["valor"] for k, v in cfg["fatores_bateria"].items()})

curva = pd.read_excel(curva_raw)
curva.columns = [str(col).strip() for col in curva.columns]
print("Colunas da curva:", list(curva.columns))
curva.head()


## 2. Tratamento

1. Ordenar por tempo e criar o passo `dt_h` (em horas).
2. **Integrar a corrente no tempo** para obter a capacidade observada e conferir contra a nominal (5,596 Ah).
3. Montar a **tabela de avaliacao** dos cenarios (energia util, autonomia, energia nao atendida, status).


In [ ]:
curva["Date_Time"] = pd.to_datetime(curva["Date_Time"])
curva = curva.sort_values("Date_Time").reset_index(drop=True)
curva["dt_h"] = curva["Date_Time"].diff().dt.total_seconds().fillna(0) / 3600.0
capacidade_Ah = float((curva["Current(A)"].abs() * curva["dt_h"]).sum())
print(f"Capacidade integrada observada: {capacidade_Ah:.4f} Ah  (~ 5,596 Ah nominais)")


### Evidencia de coleta - curva de carga (dado observado)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_curva = make_subplots(specs=[[{"secondary_y": True}]])
fig_curva.add_trace(go.Scatter(x=curva["Date_Time"], y=curva["Voltage(V)"], name="Tensao (V)"),
                    secondary_y=False)
fig_curva.add_trace(go.Scatter(x=curva["Date_Time"], y=curva["SOC"], name="SOC (0-1)"),
                    secondary_y=True)
fig_curva.update_yaxes(title_text="Tensao (V)", secondary_y=False)
fig_curva.update_yaxes(title_text="SOC (0-1)", secondary_y=True)
fig_curva.update_xaxes(title_text="Tempo de carga")
fig_curva.update_layout(title="Curva de carga 5,596 Ah - verificacao de capacidade (dado observado)",
                        template="plotly_white", height=440)
fig_curva.show()


## 3. Limpeza - as cinco verificacoes

Aplicada a **curva de carga** (dado observado). Os parametros declarados (`fatores_bateria`) tambem passam por checagem de dominio.


In [ ]:
def verificar(df, chave):
    """As cinco verificacoes de qualidade (bloco base)."""
    return {
        "1. chaves/duplicidade":
            f"{int(df.duplicated().sum())} linhas duplicadas; "
            f"{int(df.duplicated(subset=chave).sum())} duplicadas na chave {tuple(chave)}",
        "2. ausencia/codigos":
            f"{int(df.isna().sum().sum())} celulas nulas",
        "3. tipos/unidades":
            ", ".join(f"{c}={t}" for c, t in df.dtypes.astype(str).items()),
    }


def dominio_curva(df):
    return {
        "4. dominio/plausibilidade": (
            f"V {df['Voltage(V)'].min():.2f}..{df['Voltage(V)'].max():.2f}; "
            f"I {df['Current(A)'].min():.2f}..{df['Current(A)'].max():.2f}; "
            f"SOC {df['SOC'].min():.3f}..{df['SOC'].max():.3f}"),
        "5. cobertura/juncoes": (
            f"{len(df)} leituras de 1 s; dt mediano = {df['dt_h'].median() * 3600:.0f} s"),
    }


curva_verif = curva.drop(columns=["dt_h"])
relatorio = {**verificar(curva_verif, ["Date_Time"]), **dominio_curva(curva)}
pd.DataFrame(relatorio.items(), columns=["verificacao", "resultado"])


In [ ]:
parametros = cfg["fatores_bateria"]
for nome, item in parametros.items():
    assert 0 < item["valor"] <= 1, f"{nome} fora de (0, 1]"
for b in cfg["bancos"]:
    assert b["Enominal_kWh"] > 0 and b["Pmax_kW"] > 0, f"banco {b['id']} invalido"

decisoes = pd.DataFrame([
    {"problema": "Timestamps repetidos", "deteccao": "Date_Time.duplicated()",
     "afetados": int(curva["Date_Time"].duplicated().sum()),
     "regra": "manter a primeira leitura", "consequencia": "serie temporal unica"},
    {"problema": "Celulas nulas", "deteccao": "isna()",
     "afetados": int(curva.isna().sum().sum()),
     "regra": "descartar a leitura", "consequencia": "nenhuma removida"},
    {"problema": "Parametro fora de (0, 1]", "deteccao": "assert 0 < valor <= 1",
     "afetados": 0, "regra": "bloquear execucao", "consequencia": "premissas validadas"},
    {"problema": "Enom ou Pmax <= 0", "deteccao": "assert > 0",
     "afetados": 0, "regra": "bloquear execucao", "consequencia": "bancos validos"},
])

curva.to_csv(PROC / "curva_carga_5p596ah.csv", index=False)
print("Parametros e bancos validados.")
decisoes


## 4. Indicador

```
Eutilizavel   = Enom * fator_capacidade * fracao_uso * eficiencia
Edemanda      = Pcritica * duracao
autonomia     = Eutilizavel / Pcritica        (indefinida se Pcritica = 0)
Enao_atendida = max(0, Edemanda - Eutilizavel)
inviavel por potencia se Pmax < Pcritica
```


In [ ]:
fator_cap = parametros["fator_capacidade"]["valor"]
fracao_uso = parametros["fracao_uso"]["valor"]
eficiencia = parametros["eficiencia"]["valor"]


def pcritica(cargas):
    return sum(x["potencia_kW"] * x["qtd"] for x in cargas)


def energia_utilizavel(banco):
    return banco["Enominal_kWh"] * fator_cap * fracao_uso * eficiencia


def autonomia(eu, pc):
    return None if pc == 0 else eu / pc


def avaliar(cen, banco):
    pc = pcritica(cen["cargas"])
    eu = energia_utilizavel(banco)
    edem = pc * cen["duracao_h"]
    aut = autonomia(eu, pc)
    nao = max(0.0, edem - eu)
    if pc == 0:
        status = "sem_carga"
    elif banco["Pmax_kW"] < pc:
        status = "inviavel_potencia"
    elif aut is not None and aut >= cen["duracao_h"]:
        status = "atende"
    else:
        status = "nao_atende"
    return {"cenario": cen["nome"], "cenario_id": cen["id"], "banco": banco["id"],
            "Pcritica_kW": round(pc, 3), "Edemanda_kWh": round(edem, 3),
            "Eutilizavel_kWh": round(eu, 3), "Pmax_kW": banco["Pmax_kW"],
            "autonomia_h": None if aut is None else round(aut, 3),
            "Enao_atendida_kWh": round(nao, 3), "status": status}


resultados = pd.DataFrame([avaliar(cen, b) for b in cfg["bancos"] for cen in cfg["cenarios"]])
resultados


## 5. Visualizacao - V2

Barras **agrupadas** de autonomia por banco, com a **duracao requerida** marcada como linha/marcador de limiar.


In [ ]:
reais = [cen for cen in cfg["cenarios"] if cen["cargas"]]
x = [cen["nome"] for cen in reais]
cores = {"A": "#E1A100", "B": "#0B6E4F"}

fig = go.Figure()
for b in cfg["bancos"]:
    auts = [avaliar(cen, b)["autonomia_h"] for cen in reais]
    fig.add_trace(go.Bar(
        name=f"Banco {b['id']} ({b['Enominal_kWh']:.0f} kWh / {b['Pmax_kW']:.0f} kW)",
        x=x, y=auts, marker_color=cores[b["id"]],
        text=[f"{a:.1f} h" for a in auts], textposition="outside",
        hovertemplate="%{x}<br>Autonomia: %{y:.2f} h<extra></extra>"))
fig.add_trace(go.Scatter(name="Duracao requerida", x=x, y=[cen["duracao_h"] for cen in reais],
                         mode="markers+lines",
                         marker=dict(symbol="line-ew", size=24, color="#333333"),
                         line=dict(dash="dot", color="#333333")))
fig.update_yaxes(title_text="Autonomia (h)", rangemode="tozero")
fig.update_layout(title="V2. Qual cenario atende ao horizonte requerido?",
                  template="plotly_white", height=520, legend=dict(orientation="h", y=1.12))
fig.show()
fig.write_html(VISTAS / "v2_autonomia.html", include_plotlyjs="cdn")


## 6. Por que este grafico e a melhor opcao?

**Tarefa visual:** comparar **2 alternativas** (bancos A e B) atraves de **3 categorias** (cenarios) e, ao mesmo tempo, comparar cada barra com um **valor de referencia** (duracao requerida) - um teste de "atende ou nao".

- **Barras agrupadas** colocam as 2 alternativas lado a lado em cada categoria, permitindo comparacao direta par-a-par; a codificacao por **comprimento** a partir do zero e honesta.
- A **linha de limiar** ("duracao requerida") transforma o criterio de decisao em **elemento visual**: barra acima da linha = atende; abaixo = nao atende. Nao e preciso ler numeros.
- Cores distinguem as alternativas; o cinza neutro do limiar evita competir com as barras.

**Alternativas descartadas**

| Alternativa | Por que nao |
|---|---|
| Grafico de linhas | Os cenarios nao tem ordem continua; liga-los sugere tendencia inexistente. |
| Barras empilhadas | Somar autonomias de alternativas mutuamente exclusivas nao faz sentido. |
| Radar | Distorce escalas e sugere comparacao de "perfis", nao de valores. |
| Tabela pura | Correta, mas exige ler numero a numero para achar o cruzamento com o limiar. |


## 7. Leitura, decisao e limitacoes

- **Leitura:** o **Banco B** (10 kWh / 3 kW) atende aos tres cenarios; o **Banco A** (5 kWh / 1 kW) nao atende a nenhum e, no cenario de telecom, e ainda **inviavel por potencia** (`Pmax 1,0 < Pcritica 1,2`).
- **Decisao apoiada:** recomendar o Banco B; sinalizar que no Banco A a restricao dominante e a potencia, nao a energia.
- **Limite:** resultado **simulado** sob premissas declaradas (SOH 0,80; DoD 0,90; eficiencia 0,90). Nao representa medicao de uma bateria real.
